Chargement des bibliothèques

In [23]:
import pandas as pd
import numpy as np

Chargement des données

In [2]:
contrat = pd.read_csv('Contrat.csv', sep=';', low_memory = False)

In [3]:
contrat.head(5)

,idxCt,idxYear,vhImmat,sitStartDate,sitEndDate,sitExpo,drv1Age,drv1Sex,drv1DriveLicenceType,drv1DriveLicenceAge,...,id1_AssVHR,id2_AssBase,id2_Ass0km,id2_AssVHR,id3_AssBase,id3_Ass0km,id3_AssVHR,COT_AssBase,COT_Ass0km,COT_AssVHR
0,C002513884,2019,ME-6556-LY,2019-01-01,2019-12-31,1.0,39.52,H,Cond Accompagnée,18.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.01,0.0,0.0
1,C002513884,2020,ME-6556-LY,2020-01-01,2020-12-31,1.0,40.52,H,Cond Accompagnée,19.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.36,0.0,0.0
2,C002513884,2021,ME-6556-LY,2021-01-01,2021-12-31,1.0,41.52,H,Cond Accompagnée,20.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.62,0.0,0.0
3,C002513884,2022,ME-6556-LY,2022-01-01,2022-12-31,1.0,42.52,H,Cond Accompagnée,21.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.20,0.0,0.0
4,C002513884,2023,ME-6556-LY,2023-01-01,2023-12-31,1.0,43.52,H,Cond Accompagnée,22.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.00,0.0,0.0


Nettoyage des variables qualitatives

In [24]:
# 1. Création de la copie
contrat1 = contrat.copy()

# 2. Définition des groupes de variables
vars_texte = [
    'idxCt', 'vhImmat', 'ctINSEE', 
    'id1_AssBase', 'id1_Ass0km', 'id1_AssVHR', 
    'id2_AssBase', 'id2_Ass0km', 'id2_AssVHR', 
    'id3_AssBase', 'id3_Ass0km', 'id3_AssVHR'
]

vars_categorie = [
    'idxYear', 'drv1Sex', 'drv1DriveLicenceType', 'ctFrm', 
    'ctAssBase', 'ctAss0km', 'ctAssVHR', 'vhSegment', 
    'vhMarque', 'vhEnergy', 'vhGroup', 'vhClass', 
    'ctUsage', 'ctKM'
]

# Ajoutez ici les noms des colonnes qui sont des dates dans votre table contrat
vars_dates = ['sitStartDate', 'sitEndDate']

# 3. Conversion en texte (en préservant les vrais NaN)
for col in vars_texte:
    if col in contrat1.columns:
        contrat1[col] = contrat1[col].astype(str).replace('nan', np.nan)

# 4. Conversion en catégorie
for col in vars_categorie:
    if col in contrat1.columns:
        contrat1[col] = contrat1[col].astype('category')

# 5. NOUVEAU : Conversion au format Date (Datetime)
for col in vars_dates:
    if col in contrat1.columns:
        # errors='coerce' transforme les dates invalides en NaT (Not a Time)
        contrat1[col] = pd.to_datetime(contrat1[col], errors='coerce')

# 6. Traitement de l'exposition (Numérique)
if 'sitExpo' in contrat1.columns:
    contrat1['sitExpo'] = pd.to_numeric(contrat1['sitExpo'], errors='coerce')

# Vérification finale
print("--- Vérification des types dans contrat1 ---")
print(contrat1[vars_texte + vars_categorie + [d for d in vars_dates if d in contrat1.columns]].dtypes)

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_7692\903114795.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  contrat1[col] = contrat1[col].astype(str).replace('nan', np.nan)


--- Vérification des types dans contrat1 ---
idxCt                           object
vhImmat                         object
ctINSEE                         object
id1_AssBase                     object
id1_Ass0km                      object
id1_AssVHR                      object
id2_AssBase                     object
id2_Ass0km                      object
id2_AssVHR                      object
id3_AssBase                     object
id3_Ass0km                      object
id3_AssVHR                     float64
idxYear                       category
drv1Sex                       category
drv1DriveLicenceType          category
ctFrm                         category
ctAssBase                     category
ctAss0km                      category
ctAssVHR                      category
vhSegment                     category
vhMarque                      category
vhEnergy                      category
vhGroup                       category
vhClass                       category
ctUsage            

In [25]:
contrat2 = contrat1.copy()

# 2. Dictionnaire de renommage corrigé
noms_colonnes = {
    "idxCt": "identifiant_contrat",
    "idxYear": "annee_exercice",
    "vhImmat": "immatriculation_vehicule",
    "drv1Sex": "genre_conducteur",
    "drv1DriveLicenceType": "type_permis",
    "ctFrm": "formule_contrat",
    "ctAssBase": "assistance_base",
    "ctAss0km": "assistance_0km",
    "ctAssVHR": "assistance_vehicule",
    "vhSegment": "segment_vehicule",
    "vhMarque": "marque_vehicule",
    "vhEnergy": "alimentation_vehicule",
    "vhGroup": "groupe_vehicule",
    "vhClass": "classe_vehicule",
    "ctUsage": "usage_contrat",
    "ctKM": "petit_rouleur",
    "ctINSEE": "INSEE_commune",
    "sitStartDate" : "date_debut",
    "sitEndDate" : "date_fin"
}

# 3. Application du renommage sur contrat2
contrat2 = contrat2.rename(columns=noms_colonnes)

# Vérification des noms de colonnes
print(contrat2.columns)

Index(['identifiant_contrat', 'annee_exercice', 'immatriculation_vehicule',
       'date_debut', 'date_fin', 'sitExpo', 'drv1Age', 'genre_conducteur',
       'type_permis', 'drv1DriveLicenceAge', 'vhAge', 'formule_contrat',
       'assistance_base', 'assistance_0km', 'assistance_vehicule',
       'segment_vehicule', 'marque_vehicule', 'alimentation_vehicule',
       'vhWeight', 'vhDIN', 'vhValue', 'groupe_vehicule', 'classe_vehicule',
       'usage_contrat', 'petit_rouleur', 'ctDeduc', 'claimsAnt',
       'INSEE_commune', 'id1_AssBase', 'id1_Ass0km', 'id1_AssVHR',
       'id2_AssBase', 'id2_Ass0km', 'id2_AssVHR', 'id3_AssBase', 'id3_Ass0km',
       'id3_AssVHR', 'COT_AssBase', 'COT_Ass0km', 'COT_AssVHR'],
      dtype='object')


In [26]:
# Liste des variables à inspecter (noms renommés)
vars_a_inspecter = [
    'genre_conducteur', 'type_permis', 'formule_contrat', 
    'assistance_base', 'assistance_0km', 'assistance_vehicule', 
    'segment_vehicule', 'marque_vehicule', 'alimentation_vehicule', 
    'groupe_vehicule', 'classe_vehicule', 'usage_contrat', 
    'petit_rouleur'
]

# Affichage des valeurs uniques pour chaque variable
for var in vars_a_inspecter:
    if var in contrat2.columns:
        print(f"--- Valeurs pour '{var}' ---")
        print(contrat2[var].unique())
        print("\n")

--- Valeurs pour 'genre_conducteur' ---
['H', 'F']
Categories (2, object): ['F', 'H']


--- Valeurs pour 'type_permis' ---
['Cond Accompagnée', 'Traditionnel']
Categories (2, object): ['Cond Accompagnée', 'Traditionnel']


--- Valeurs pour 'formule_contrat' ---
['Med2', 'Mini', 'Maxi', 'Med1']
Categories (4, object): ['Maxi', 'Med1', 'Med2', 'Mini']


--- Valeurs pour 'assistance_base' ---
[1]
Categories (1, int64): [1]


--- Valeurs pour 'assistance_0km' ---
[0, 1]
Categories (2, int64): [0, 1]


--- Valeurs pour 'assistance_vehicule' ---
[0, 1]
Categories (2, int64): [0, 1]


--- Valeurs pour 'segment_vehicule' ---
['Familiale', 'Compacte', 'Citadine', 'SUV']
Categories (4, object): ['Citadine', 'Compacte', 'Familiale', 'SUV']


--- Valeurs pour 'marque_vehicule' ---
['Peugeot', 'Audi', 'Citroen', 'Mercedes Benz', 'Volkswagen', ..., 'Nissan', 'Dacia', 'Fiat', 'Ford', 'BMW']
Length: 14
Categories (14, object): ['Audi', 'Autre', 'BMW', 'Citroen', ..., 'Renault', 'Seat', 'Toyota', 'Volk

In [27]:
# 1. Création de contrat3 à partir de contrat2
contrat3 = contrat2.copy()

# 2. Liste des variables dont on va modifier les modalités
vars_to_rename = [
    'genre_conducteur', 'type_permis', 'assistance_base', 
    'assistance_0km', 'assistance_vehicule', 'usage_contrat', 'petit_rouleur'
]

# 3. Conversion temporaire en string pour effectuer les remplacements librement
for col in vars_to_rename:
    if col in contrat3.columns:
        contrat3[col] = contrat3[col].astype(str)

# 4. Application des remplacements de modalités
contrat3['genre_conducteur'] = contrat3['genre_conducteur'].replace({"H": "Homme", "F": "Femme"})
contrat3['type_permis'] = contrat3['type_permis'].replace({"Cond Accompagnée": "Conduite accompagnée"})
contrat3['assistance_base'] = contrat3['assistance_base'].replace({"1": "oui"})
contrat3['assistance_0km'] = contrat3['assistance_0km'].replace({"0": "non", "1": "oui"})
contrat3['assistance_vehicule'] = contrat3['assistance_vehicule'].replace({"0": "non", "1": "oui"})
contrat3['usage_contrat'] = contrat3['usage_contrat'].replace({"Pri": "privé", "Pro": "professionnel"})
contrat3['petit_rouleur'] = contrat3['petit_rouleur'].replace({"N": "non", "O": "oui", "0": "non", "1": "oui"})

# 5. Conversion finale en 'category' pour l'optimisation mémoire
for col in vars_to_rename:
    if col in contrat3.columns:
        contrat3[col] = contrat3[col].astype('category')

# Vérification rapide du résultat
print(contrat3[vars_to_rename].head())

  genre_conducteur           type_permis assistance_base assistance_0km  \
0            Homme  Conduite accompagnée             oui            non   
1            Homme  Conduite accompagnée             oui            non   
2            Homme  Conduite accompagnée             oui            non   
3            Homme  Conduite accompagnée             oui            non   
4            Homme  Conduite accompagnée             oui            non   

  assistance_vehicule  usage_contrat petit_rouleur  
0                 non  professionnel           non  
1                 non  professionnel           non  
2                 non  professionnel           non  
3                 non  professionnel           non  
4                 non  professionnel           non  


In [28]:
# 1. Création de contrat4 à partir de contrat3
contrat4 = contrat3.copy()

# 2. Conversion en numérique pour pouvoir tester la condition (1900 < année < 2026)
contrat4['annee_exercice'] = pd.to_numeric(contrat4['annee_exercice'], errors='coerce')

# 3. Application du filtre
# On ne garde que les lignes où l'année est comprise entre 1900 et 2026 et n'est pas vide
condition_annee = (contrat4['annee_exercice'] >= 1900) & (contrat4['annee_exercice'] <= 2026)
contrat4 = contrat4[condition_annee].copy()

# 4. Conversion finale en 'category' pour l'optimisation
contrat4['annee_exercice'] = contrat4['annee_exercice'].astype(int).astype('category')

# Vérification du nombre de lignes supprimées
lignes_supprimees = len(contrat3) - len(contrat4)
print(f"Nombre de lignes avec une année invalide supprimées : {lignes_supprimees}")
print(f"Années présentes dans contrat4 : {contrat4['annee_exercice'].unique().tolist()}")

Nombre de lignes avec une année invalide supprimées : 0
Années présentes dans contrat4 : [2019, 2020, 2021, 2022, 2023]


Nettoyage des données quantitatives

In [29]:
# 1. Création de contrat5 à partir de contrat4
contrat5 = contrat4.copy()

# 2. Dictionnaire de renommage pour les variables quantitatives
noms_quant = {
    "sitExpo": "exposition",
    "drv1Age": "age_conducteur",
    "drv1DriveLicenceAge": "anciennete_permis",
    "vhAge": "age_vehicule",
    "vhWeight": "poids_vehicule",
    "vhDIN": "puissance_vehicule",
    "vhValue": "valeur_vehicule",
    "ctDeduc": "franchise",
    "claimsAnt": "sinistres_anterieurs",
    "COT_AssBase": "cotisation_base",
    "COT_Ass0km": "cotisation_0km",
    "COT_AssVHR": "cotisation_vhr"
}

# 3. Application du renommage
contrat5 = contrat5.rename(columns=noms_quant)

# 4. Conversion forcée en numérique (float) pour toutes ces variables
vars_quant_renommees = list(noms_quant.values())

for col in vars_quant_renommees:
    contrat5[col] = pd.to_numeric(contrat5[col], errors='coerce')

# 5. Traitement des valeurs aberrantes (Nettoyage logique)
# Age conducteur : entre 17 et 100 ans (le reste devient NaN)
contrat5.loc[(contrat5['age_conducteur'] < 17) | (contrat5['age_conducteur'] > 100), 'age_conducteur'] = np.nan

# Ancienneté permis : ne peut pas être négative ou supérieure à l'âge
contrat5.loc[contrat5['anciennete_permis'] < 0, 'anciennete_permis'] = 0

# Poids véhicule : un véhicule léger fait au moins 400kg
contrat5.loc[contrat5['poids_vehicule'] < 400, 'poids_vehicule'] = np.nan

# Valeurs financières (cotisations, franchises, valeur vh) : doivent être positives
cols_financieres = ['cotisation_base', 'cotisation_0km', 'cotisation_vhr', 'franchise', 'valeur_vehicule']
for col in cols_financieres:
    contrat5.loc[contrat5[col] < 0, col] = np.nan

# Vérification statistique
print("Résumé des variables quantitatives après nettoyage :")
print(contrat5[vars_quant_renommees].describe())

Résumé des variables quantitatives après nettoyage :
          exposition  age_conducteur  anciennete_permis   age_vehicule  \
count  301437.000000   301437.000000      301437.000000  301437.000000   
mean        0.966932       44.693525          24.737300      11.485966   
std         0.144753       13.685469          13.659565       4.879809   
min         0.002732       18.500000          -0.000000       0.000000   
25%         1.000000       33.870000          13.860000       8.180000   
50%         1.000000       43.820000          23.850000      11.640000   
75%         1.000000       54.570000          34.580000      14.960000   
max         1.000000       79.000000          60.860000      24.000000   

       poids_vehicule  puissance_vehicule  valeur_vehicule      franchise  \
count   301437.000000       301437.000000    301437.000000  301437.000000   
mean      1387.545716          132.558810     11466.762802     289.778959   
std        308.583216           47.501897      61

Traitement des doublons

In [30]:
# 1. Création de contrat7 à partir de contrat6
contrat6 = contrat5.copy()

# 2. Comptage avant suppression
nb_lignes_avant = len(contrat6)

# 3. Suppression des doublons stricts
# Sans l'argument 'subset', pandas compare toutes les colonnes
contrat6 = contrat6.drop_duplicates(keep='first')

# 4. Calcul du nombre de lignes supprimées
nb_doublons_stricts = nb_lignes_avant - len(contrat6)

print(f"Nombre de doublons stricts supprimés : {nb_doublons_stricts}")
print(f"Taille de la base après nettoyage des doublons stricts : {len(contrat6)} lignes")

Nombre de doublons stricts supprimés : 0
Taille de la base après nettoyage des doublons stricts : 301437 lignes


Traitement des valeurs manquantes

In [31]:
# Calcul du nombre de valeurs manquantes par colonne dans contrat8
missing_count = contrat6.isnull().sum()

# Calcul du pourcentage pour une meilleure lecture
missing_percent = (contrat6.isnull().sum() / len(contrat6)) * 100

# Création d'un tableau récapitulatif
df_missing = pd.DataFrame({
    'Valeurs Manquantes': missing_count,
    'Pourcentage (%)': missing_percent
})

# On n'affiche que les colonnes qui ont au moins une valeur manquante (triées par importance)
print("État des lieux des valeurs manquantes dans contrat6 :")
print(df_missing[df_missing['Valeurs Manquantes'] > 0].sort_values(by='Valeurs Manquantes', ascending=False))

État des lieux des valeurs manquantes dans contrat6 :
             Valeurs Manquantes  Pourcentage (%)
id3_AssVHR               301437       100.000000
id3_Ass0km               301435        99.999337
id2_AssVHR               301403        99.988721
id3_AssBase              301369        99.977441
id2_Ass0km               301335        99.966162
id1_AssVHR               300472        99.679867
id2_AssBase              299885        99.485133
id1_Ass0km               298994        99.189549
id1_AssBase              272856        90.518417


Ces valeurs manquantes sont tout à fait normales : les clients ne souscrivent pas tous à tous les types de contrat.